# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Let's review the available record sets, fields, and their `@id` values.

> **Note:** All entities (record sets, fields, columns) are referenced by their `@id` to guarantee correct programmatic usage.

In [ ]:
# List available record sets:
record_sets = list(dataset.record_sets())
print("Available Record Sets (by @id):\n------------------------------")
for rs in record_sets:
    print(f"@id: {rs['@id']}  |  name: {rs.get('name', 'N/A')}")

# For each record set, list its fields (by @id):
print("\nFields for each Record Set:\n---------------------------")
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    # Some schemas have field as dict if single field, else list
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        if isinstance(fld, dict):
            print(f"  Field @id: {fld.get('@id', '')}  |  name: {fld.get('name', 'N/A')}")
        else:
            print(f"  Field @id: {fld}")

## 3. Data Extraction

Let's extract tabular data from a selected record set. For this dataset, the main record set contains clinicopathological and molecular records. We'll use its `@id` as reference.

> Use the record set and field `@id`s printed above for precise field extraction.

In [ ]:
# Identify record sets (referenced by @id, typically only one for small clinical tables):
record_sets = list(dataset.record_sets())

# Choose the main tabular record set for analysis;
# Replace the @id below if your dataset has more than one or names differ
# We'll programmatically extract all available record set @id's for demonstration, but you can select a subset
record_set_ids = [rs['@id'] for rs in record_sets]

# Load each record set as a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print columns (fields) of the main record set (use first for now)
main_record_set_id = record_set_ids[0]
print(f"Columns in record set {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform typical data processing steps—filtering, normalizing, and grouping—referencing columns by their `@id` as required.

For demonstration, we'll use a numeric field (for example, a field representing 'age' or a numerical measurement). Adjust to your preferred field if your field names differ.

In [ ]:
# Inspect columns for numeric fields
df = dataframes[main_record_set_id]
print("Columns available for EDA:", df.columns.tolist())

# Pick a numeric field by its @id (replace with the correct one if needed)
# Let's try to pick 'age' or a similar field; you may need to adjust the field ID.
numeric_field_candidates = [col for col in df.columns if 'age' in col.lower() or df[col].dtype.kind in 'fi']
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    # fallback: pick first float/int field
    numeric_field = next((col for col in df.columns if df[col].dtype.kind in 'fi'), df.columns[0])
print(f"Using numeric field for analysis: {numeric_field}")

# Threshold filtering (using a value appropriate for 'age'; adjust as needed)
threshold = 50
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Select a grouping field (categorical)
group_field_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
if group_field_candidates:
    group_field = group_field_candidates[0]
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
    print(grouped_df.head())
else:
    print("No categorical field available for grouping.")

## 5. Visualization

Let's visualize the distribution of the chosen numeric field and its normalization. We'll use matplotlib for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field} (by @id)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If normalized available
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=15, color='darkcyan')
    plt.title(f"Normalized Distribution of {numeric_field} (filtered, by @id)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion

- Using `mlcroissant`, we loaded and explored tabular clinical data from the FAIR² Croissant package, referencing all dataset entities by their `@id`.
- The dataset provides rich demographic and clinical details, with fields easily accessible for filtering, transformation, and grouping by programmatic `@id`.
- Visualizations facilitate insights into field distributions and group comparisons for informed biomedical analysis.

For advanced use, further process and model these data fields, keeping strict reference to their Croissant schema `@id` throughout all analyses.